In [2]:
from dataclasses import dataclass
import os
from openai import OpenAI

@dataclass(frozen=True)
class Provider:
    """ One provider to relaiably route all requests around inference providers"""
    name:str
    env_var:str
    is_free:bool
    base_url:str|None
    model:str

PROVIDERS = [
    Provider("OpenAI","OPENAI_API_KEY",False,None,"gpt-4o-mini"),
    Provider("Groq", "GROQ_API_KEY", True, "https://api.groq.com/openai/v1", "openai/gpt-oss-120b"),
]

def select_provider()->Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider
    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")

def build_client(provider:Provider)->OpenAI:
    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(api_key=api_key,model=provider.model)

def have_any_key()->bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")

Found a provider key.
